In [214]:
import yaml
import copy
from datetime import datetime,timezone,timedelta


In [215]:
def load_yaml(filepath: str) -> dict:
    with open(filepath, "r", encoding="utf-8") as f:
        return yaml.safe_load(f)

In [216]:
config          = load_yaml(r"C:\Users\TunKedsaro\Desktop\CVResume\src\config\weight.yaml") 

In [217]:
op1 = {
    'section': 'Profile',
    'scores': {
        'ContentQuality': {
            'score': 1,
            'feedback': "xxx"},
        'Completeness': {
            'score': 0,
            'feedback': "yyy"}
        },
    'session_feedback': 'zzz'
}

In [218]:
op3 = {
    'section': 'Education',
    'scores': {
        'RoleRelevance': {
            'score': 5,
            'feedback': 'abc'},
        'Completeness': {
        'score': 5,
        'feedback': "def"}
        },
    'session_feedback': 'ghi'
}

In [219]:
# Now score from llm is 1-5 but when It's return 0 It's mean there are no
# that section so do calculate it 

In [232]:
def aggregate(llm_output:dict,verbose=0):
    '''
    convert, reshape, transform output format that We got from llm
    '''
    # print(f"llm_output ->\n{llm_output}")
    llm_output       = llm_output                  # op
    section_name     = llm_output["section"]       # Get section
    section_scores   = config["weights"][section_name]  # config["weights"][section_key][criteria]
    ddict = {} 
    total_score = 0.0
    full_score  = 0.0
    scores_copy = copy.deepcopy(llm_output["scores"])  # Protect multiple mutation when we run more than one time
    for criteria,body in scores_copy.items():
        raw_score_from_llm = body["score"]             # score ดิบๆ ที่ออกมาจาก LLM
        if raw_score_from_llm == 0:
            full_score_from_config = 0
        else:
            full_score_from_config = section_scores[criteria]   # weight ที่ set ใน weight.yaml เพื่อ scale เต็มๆ
        scored = raw_score_from_llm / 5 * full_score_from_config          # raw / normalize max score * weight scale
        body["score"] = scored
        ddict[criteria] = body
        total_score = total_score + scored
        full_score  = full_score  + full_score_from_config
        if verbose:
            print(f"criteria->{criteria}")
            print(f"body->{body}")
            print(f" - raw -> {raw}") 
            print(f" - w   -> {w}")
            print(f" - weighted -> {weighted}")  
            print(f" - body['score'] -> {body['score']}")  
            print(f"total -> {total}")
            print()
    return {
            "section": section_name,
            "total_score":total_score,
            "full_score":full_score,
            "scores":ddict,
            "session_feedback":llm_output['session_feedback']
        }

In [233]:
print("s1")
s1 = aggregate(op1)
print("s2")
s2 = aggregate(op3)

s1
s2


In [234]:
s1

{'section': 'Profile',
 'total_score': 2.0,
 'full_score': 10.0,
 'scores': {'ContentQuality': {'score': 2.0, 'feedback': 'xxx'},
  'Completeness': {'score': 0.0, 'feedback': 'yyy'}},
 'session_feedback': 'zzz'}

In [223]:
s2

{'section': 'Education',
 'total_score': 20.0,
 'full_score': 20.0,
 'scores': {'RoleRelevance': {'score': 10.0, 'feedback': 'abc'},
  'Completeness': {'score': 10.0, 'feedback': 'def'}},
 'session_feedback': 'ghi'}

<hr>

In [235]:
section_outputs = [s1,s2]
timestamp       = str(datetime.now(tz=(timezone(timedelta(hours=7)))))
model_config    = load_yaml(r"C:\Users\TunKedsaro\Desktop\CVResume\src\config\model.yaml")     # should include model name
weight_config   = load_yaml(r"C:\Users\TunKedsaro\Desktop\CVResume\src\config\weight.yaml")    # includes weights + version
prompt_config   = load_yaml(r"C:\Users\TunKedsaro\Desktop\CVResume\src\config\prompt.yaml") 
config_lang     = prompt_config['Language_output_style']["en"]

In [236]:
section_outputs

[{'section': 'Profile',
  'total_score': 2.0,
  'full_score': 10.0,
  'scores': {'ContentQuality': {'score': 2.0, 'feedback': 'xxx'},
   'Completeness': {'score': 0.0, 'feedback': 'yyy'}},
  'session_feedback': 'zzz'},
 {'section': 'Education',
  'total_score': 20.0,
  'full_score': 20.0,
  'scores': {'RoleRelevance': {'score': 10.0, 'feedback': 'abc'},
   'Completeness': {'score': 10.0, 'feedback': 'def'}},
  'session_feedback': 'ghi'}]

In [226]:
every_section_score_total = 0
every_section_full_score_total = 0

for section_data in section_outputs:
    section_name                   = section_data["section"]
    section_total_score            = section_data["total_score"]
    section_full_score             = section_data["full_score"]
    every_section_score_total      = every_section_score_total + section_total_score
    every_section_full_score_total = every_section_full_score_total + section_full_score

In [227]:
every_section_score_total

22.0

In [228]:
every_section_full_score_total

30.0

<hr>

In [229]:
# 90–100 /100 -> A
# 80–89  /100 -> B
# 70–79  /100 -> C
# 60–69  /100 -> D
# 00–49  /100 -> F

def normalize_score(point_score,full_score):
    print(f"Point_score -> {point_score}")
    print(f"full_score  -> {full_score}")
    normalize_score = (point_score/full_score) * 100
    print(f"normalize_score -> {normalize_score}/100")
    return normalize_score

def score_to_grade(score: float) -> str:
    if 90 <= score <= 100:
        return "A"
    elif 80 <= score < 90:
        return "B"
    elif 70 <= score < 80:
        return "C"
    elif 60 <= score < 70:
        return "D"
    elif 0 <= score < 60:
        return "F"
    else:
        return "Gradding error"

# # 40/40*100 -> 100.0
# normalized_score = normalize_score(40,40)
# score_to_grade(normalized_score)

# # 40/40*100 -> 100.0
# normalized_score = normalize_score(50,100)
# score_to_grade(normalized_score)

# # 30/40 * 100
# normalize_score(30,40)
# score_to_grade(normalized_score)

In [230]:
# 40/40*100 -> 100.0
normalized_score = normalize_score(every_section_score_total,every_section_full_score_total)
score_to_grade(normalized_score)

Point_score -> 22.0
full_score  -> 30.0
normalize_score -> 73.33333333333333/100


'C'

Report 00
- คอมมีปัญหาหน้าจอดับ

Report 01
- Ticket ของ week ที่แล้วเสร็จหมดแล้วตอนนี้อยู่ Branch dev
- ตอนนี้ทุกอย่างเป็น Promptbuilder -> PromptBuilder + Session,Global feedback + PromptSplit = BasePromptBuilder
- ยิง 300 - 400 ครั้ง error 1 ครับ แต่อันนี้ไม่ชัวเพราะ ขึ้นอยู่กับ LLM เลยยิ่งใข้นานยิ่งเจอ Bug
- PowerBI report
- ทดสอบทั้งหมด 3 branch branch ละ 100 example = 300 examples
- ตอนนี้ไม่ได้ Deploy ขึ้นเพราะอยากสร้าง Endpoint ใหม่

In [ ]:
Report 02
- Rethink? -> Rethink
- temperature? -> Rethink

Question 01
- input ของแต่ละ CV,Resume เนี้ยอยู่ในระบบ Web ใช่ไหม ?
     - การ Detect และตรวจสอบว่า User ขาดอะไรให้ทำเป็นแบบ JSON + Pure python bc เร็วกว่า LLM ไม่งั้นจะเป็นการเพิ่ม LLM ไปอีก 1 ชั้น
- ถ้าอย่างนั้นมันต้องมีการกำหนด Template กลางระหว่าง Dev & Data
- อันนี้เป็นตัวอย่างของ CV Resume
- ถ้านับรวมตรงมีโอกาสไม่สมบูรณ์เพราะต้องไป align with dev team

Question 02 : Output
- ตอนนี้สิ่งที่ออกมาจาก LLM คือ

In [ ]:
{
  "response": {
    "conclution": {
      "final_resume_score": 23.2,
      "section_contribution": {
        "Profile": {
          "section_total": 8,
          "section_weight": 0.1,
          "contribution": 0.8
        },
        "Summary": {
          "section_total": 48,
          "section_weight": 0.1,
          "contribution": 4.8
        },
        "Education": {
          "section_total": 20,
          "section_weight": 0.2,
          "contribution": 4
        },
        "Experience": {
          "section_total": 42,
          "section_weight": 0.2,
          "contribution": 8.4
        },
        "Activities": {
          "section_total": 0,
          "section_weight": 0.2,
          "contribution": 0
        },
        "Skills": {
          "section_total": 26,
          "section_weight": 0.2,
          "contribution": 5.2
        }
      },
      "globalfeedback": {
        "response": "Strong resume, but boost impact by quantifying achievements and using powerful action verbs consistently. Refine summary goals."
      }
    },
    "section_detail": {
      "Profile": {
        "total_score": 8,
        "scores": {
          "ContentQuality": {
            "score": 2,
            "feedback": "The profile section lacks an explicit professional title, hindering the immediate establishment of the candidate's identity."
          },
          "Completeness": {
            "score": 6,
            "feedback": "Essential contact details and relevant links are included, but a clear professional title is missing from the profile header."
          }
        },
        "session_feedback": "The profile successfully provides complete contact details and valuable professional links. However, the absence of a clear professional title in the header diminishes the immediate presentation of the candidate's professional identity for the Data Scientist role."
      },
      "Summary": {
        "total_score": 48,
        "scores": {
          "RoleRelevance": {
            "score": 10,
            "feedback": "The summary strongly aligns with a Data Scientist role, emphasizing relevant skills and business impact."
          },
          "Length": {
            "score": 10,
            "feedback": "The summary is concise at two sentences, providing good information density without redundancy."
          },
          "Grammar": {
            "score": 10,
            "feedback": "The writing is clear, professional, and free of grammatical errors, making it easy to read."
          },
          "ContentQuality": {
            "score": 8,
            "feedback": "It highlights specific focuses and business impact effectively, though some phrasing is slightly generic."
          },
          "Completeness": {
            "score": 10,
            "feedback": "The summary clearly establishes identity, highlights relevant expertise, and communicates a strong value proposition."
          }
        },
        "session_feedback": "The summary effectively positions the candidate for a Data Scientist role by clearly stating their expertise in data analysis, experiments, and driving business growth. Its concise nature and strong relevance enhance readability and appeal, despite some generic phrasing."
      },
      "Education": {
        "total_score": 20,
        "scores": {
          "RoleRelevance": {
            "score": 10,
            "feedback": "The educational background strongly supports the knowledge and skills expected for the target role."
          },
          "Completeness": {
            "score": 10,
            "feedback": "Clearly presents institution, degree, field of study, and dates, allowing the reader to easily understand the candidate's education."
          }
        },
        "session_feedback": "The education section is outstanding, featuring multiple highly relevant master's degrees that directly align with data science knowledge and skills. It is also very complete, providing strong academic credentials for the role."
      },
      "Experience": {
        "total_score": 42,
        "scores": {
          "RoleRelevance": {
            "score": 6,
            "feedback": "Two Data Scientist roles show strong relevance, but the majority of experience is in sales and tutoring, reducing overall alignment."
          },
          "Length": {
            "score": 10,
            "feedback": "Bullet points are concise and sufficiently detailed, conveying key information effectively without being verbose."
          },
          "Grammar": {
            "score": 10,
            "feedback": "The writing is clear, professional, and grammatically correct, ensuring strong readability throughout the section."
          },
          "ContentQuality": {
            "score": 6,
            "feedback": "Data Scientist roles detail actions and methods effectively but lack clear, quantifiable impact from the projects."
          },
          "Completeness": {
            "score": 10,
            "feedback": "All key elements are present, including role, company, duration, responsibilities, and outcomes, making experience easy to understand."
          }
        },
        "session_feedback": "This section is clear and well-structured. The Data Scientist roles demonstrate relevant skills and methods but are brief, lacking quantifiable impact. The extensive non-DS experience suggests a career transition, potentially limiting immediate fit for core Data Scientist positions."
      },
      "Activities": {
        "total_score": 0,
        "scores": {
          "Length": {
            "score": 0,
            "feedback": "The 'Activities' section is missing from the resume, so no content is available for evaluation."
          },
          "Grammar": {
            "score": 0,
            "feedback": "The 'Activities' section is absent from the resume, preventing any grammatical assessment."
          },
          "ContentQuality": {
            "score": 0,
            "feedback": "With no 'Activities' section present, content quality cannot be assessed."
          },
          "Completeness": {
            "score": 0,
            "feedback": "The 'Activities' section is entirely missing, precluding any completeness evaluation."
          }
        },
        "session_feedback": "The resume lacks an 'Activities' section, making it impossible to evaluate extracurricular involvement for the Data Scientist role."
      },
      "Skills": {
        "total_score": 26,
        "scores": {
          "RoleRelevance": {
            "score": 10,
            "feedback": "Skills are highly relevant and directly support the target data scientist role."
          },
          "Length": {
            "score": 6,
            "feedback": "The list is quite extensive; it could be more concise to enhance scannability."
          },
          "Completeness": {
            "score": 10,
            "feedback": "Provides a well-organized and comprehensive list of skills that clearly represents the candidate's capabilities."
          }
        },
        "session_feedback": "This skills section effectively positions the candidate with a highly relevant and well-organized comprehensive list of core data science competencies. While the overall length could be slightly refined by removing redundant language entries or very basic tools, the strong alignment with the target role and clear categorization demonstrate strong capability."
      }
    },
    "metadata": {
      "model_name": "gemini-2.5-flash",
      "timestamp": "2026-01-12 12:45:11.773764+07:00",
      "weights_version": "weights_v1",
      "prompt_version": "prompt_v2"
    }
  },
  "response_time": "27.06838 s",
  "estimated_cost_thd": "0.25676 ฿"
}

Question 03 : Adaptive score
- Figma process workflow
- มีการรื้อหลายจุด และ Modify
    - ?มีการเพิ่ม คลาสใหม่
    - aggregator.py เป็นหลัก โดย สิ่งที่ออกมาจาก LLM ->
        - 0 : ไม่มี
        - 1-5 : grade scale
      ตอนนี้คือสร้างสิ่งนึ่งที่คอยเก็บ score ที่เป็นไปได้ทั้งหมดว่า
    - Jira compare
    - การทำให้มัน adaptive เนี้ยจะเอาระดับไหน 
        - Section  (x)
        - Criteria ( )
        - Item     ( )
    - ถ้าระดับ Criteria 
    - ถ้าระดับ Item

In [ ]:
{
    "Profile": {
        "name": "Juan Jose Carin",
        "email": "juanjose.carin@gmail.com",
        "phone": "650-336-4590",
        "linkedin": "linkedin.com/in/juanjosecarin",
        "jobdb_link": "-",
        "portfolio_link": "juanjocarin.github.io"
    },
    "Summary": {
        "has_summary": "Yes",
        "summary_points": [
            "Passionate about data analysis and experiments, mainly focused on user behavior, experience, and engagement.",
            "Solid background in data science and statistics, with extensive experience using data insights to drive business growth."
        ]
    },
    "Education": [
        {
            "institution": "University of California, Berkeley",
            "degree": "Master of Information and Data Science",
            "dates": "2016",
            "gpa": "3.93",
            "honors": "Relevant courses: Machine Learning; Machine Learning at Scale; Storing and Retrieving Data; Field Experiments; Applied Regression and Time Series Analysis; Exploring and Analyzing Data; Data Visualization and Communication; Research Design and Applications for Data Analysis."
        },
        {
            "institution": "Universidad Politécnica de Madrid",
            "degree": "M.S. in Statistical and Computational Information Processing",
            "dates": "2014",
            "gpa": "3.69",
            "honors": "Relevant courses: Data Mining; Multivariate Analysis; Time Series; Neural Networks and Statistical Learning; Regression and Prediction Methods; Optimization Techniques; Monte Carlo Techniques; Numerical Methods in Finance; Stochastic Models in Finance; Bayesian Networks."
        },
        {
            "institution": "Universidad Politécnica de Madrid",
            "degree": "M.S. in Telecommunication Engineering",
            "dates": "2005",
            "gpa": "3.03",
            "honors": "Focus Area: Radio communication systems (radar and mobile). Fellowship: First year at University, due to Honors obtained last year at high school."
        }
    ],
    "Experience": [
        {
            "title": "Data Scientist",
            "company": "CONENTO (Madrid, Spain) — working remotely",
            "dates": "Jan 2016 - Mar 2016",
            "description": [
                "Designed and implemented the ETL pipeline for a predictive model of traffic on the main roads in eastern Spain (project for the Spanish government).","Automated scripts in R to extract, transform, clean (including anomaly detection), and load into MySQL data from multiple sources (road traffic sensors, accidents, road works, weather)."
            ]
        },
        {
            "title": "Data Scientist",
            "company": "CONENTO (Madrid, Spain)",
            "dates": "Jun 2014 - Sep 2014",
            "description": [
                "Designed an experiment for Google Spain (conducted in Oct 2014) to measure the impact of YouTube ads on the sales of a car manufacturer's dealer network.","Used a matched-pair, cluster-randomized design selecting test/control groups from 50+ cities based on sales-wise similarity over time (using wavelets and R)."
            ]
        },
        {
            "title": "Head of Sales, Spain & Portugal - Test & Measurement dept.",
            "company": "YOKOGAWA (Madrid, Spain)",
            "dates": "Feb 2009 - Aug 2013",
            "description": [
                "Applied analysis of sales and market trends to decide the direction of the department.","Led a team of 7 people.","Increased revenue by 6.3%, gross profit by 4.2%, and operating income by 146%; achieved a 30% ratio of new customers (3x growth) by entering new markets and improving customer service and training."
            ]
        },
        {
            "title": "Sales Engineer - Test & Measurement dept.",
            "company": "YOKOGAWA (Madrid, Spain)",
            "dates": "Apr 2008 - Jan 2009",
            "description": [
                "Promoted to head of sales after 5 months leading the sales team."
            ]
        },
        {
            "title": "Sales & Application Engineer",
            "company": "AYSCOM (Madrid, Spain)",
            "dates": "Sep 2004 - Mar 2008",
            "description": ["Exceeded sales target every year from 2005 to 2007; achieved 60% of the target in the first 3 months of 2008."]
        },
        {
            "title": "Tutor of Differential & Integral Calculus, Physics, and Digital Electronic Circuits",
            "company": "ACADEMIA UNIVERSITARIA (Madrid, Spain)",
            "dates": "Jul 2002 - Jun 2004",
            "description": [
                "Highest-rated professor in student surveys in 4 of the 6 terms.",
                "Increased ratio of students passing the course by 25%."
            ]
        }
    ],
    "Activities": [
        {
            "name": "LLM Evaluation Harness (Internal Tooling)",
            "details": "Built a golden-dataset based evaluation harness (rubric scoring, regression checks, release gates) for RAG and resume-evaluation prompts; improved release confidence and reduced regressions by ~60%.",
            "tech_stack": ["Python", "pytest", "BigQuery", "RAG"]
        },
        {
            "name": "Knowledge Sharing / Tech Talks",
            "details": "Ran internal workshops on orchestrator architecture, API reliability patterns, and production LLM practices (prompt versioning, caching, monitoring)."
        },
        {
            "name": "Open-source / Internal Libraries",
            "details": "Contributed reusable utilities for schema validation and JSON naming conversion; reduced integration bugs and improved developer onboarding."
            }
        ],
    "Skills": {
        "technical": [
            "Data Analysis","Statistics","Experiment Design","ETL","Anomaly Detection","MySQL","Machine Learning","Data Visualization","Big Data"
        ],
        "soft_skills": [
            "Team Leadership","Customer Service Improvement","Training & Mentoring"
        ],
        "tools_with_levels": [
            {"tool": "Python","level": "proficient"},
            {"tool": "R","level": "proficient"},
            {"tool": "SQL","level": "proficient"},
            {"tool": "Hadoop","level": "proficient"},
            {"tool": "Hive","level": "proficient"},
            {"tool": "MrJob","level": "proficient"},
            {"tool": "Tableau","level": "proficient"},
            {"tool": "Git","level": "proficient"},
            {"tool": "AWS","level": "proficient"},
            {"tool": "Spark","level": "intermediate"},
            {"tool": "Storm","level": "intermediate"},
            {"tool": "Bash","level": "intermediate"},
            {"tool": "SPSS","level": "intermediate"},
            {"tool": "SAS","level": "intermediate"},
            {"tool": "Matlab","level": "intermediate"},
            {"tool": "EViews","level": "basic"},
            {"tool": "Demetra+","level": "basic"},
            {"tool": "D3.js","level": "basic"},
            {"tool": "Gephi","level": "basic"},
            {"tool": "Neo4j","level": "basic"},
            {"tool": "QGIS","level": "basic"}
        ],
        "languages": [
            {"language": "Python","level": "proficient"},
            {"language": "R","level": "proficient"},
            {"language": "SQL","level": "proficient"},
            {"language": "Bash","level": "intermediate"},
            {"language": "Matlab","level": "intermediate"}
        ]
    }
}

Question 04 : New output
- ตอนนี้ที่ออกแบบเอาไว้คือมีการ ตัด เกรด 'A','B','C','D','F'
- ตอนนี้ตั้งใจว่าจะลบพวก section_weight ยุ่งยาก + ไม่ต้องใช้ ตอนนี้ทุกอย่างใช้เป็น
    - (Get_score/Full_score)x100 -> ตัดเกรดเทียบ 100 -> Grade
- Output อาจจะเปลี่ยน



<hr>

### Aprove

- Retry -> 3 -> 1 (night)

- Update orchestrator (Thus)

- Report 02 (night)
    - Rethink? -> Rethink
    - temperature? -> Rethink
    - 2.5 Flash-Lite
    - ถ้าเร็วทำ ถ้ายาวไปก่อน
    - models,InputJson,Output,Timeduration (สัก 1 session) (Today!!!)
    
- Confirm API spec สำหรับ Global CV,Resume

- เรื่อง Role ต้อง Blank (Now)
- ถ้าหากว่าไม่ใส่ Role มันคือ N/A -> Employee
- ถ้าหากว่าไม่ใส่ Role มันคือ N/A และมันต้องตัด Role relevance ทิ้ง
